# Guardrailed Multi-Agent System


## Installing dependencies



In [57]:
!pip install -q -U --force-reinstall --no-cache-dir "Pillow>=11.3.0"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 110.5 MB/s eta 0:00:00


In [58]:
# Install all libraries required for the multi-agent workflow.
#
# crewai[litellm] -> Creates and runs the Planner, Executor, and Reviewer agents.
# langgraph      -> Builds the stateful agent graph and controls routing/retries.
# pydantic-ai    -> Validates agent responses against strict Pydantic schemas.
# mem0ai         -> Provides persistent semantic memory across executions.
# groq           -> Provides access to Groq-hosted LLMs.
# sentence-transformers -> Provides the Hugging Face embedding model.
# chromadb       -> Stores vector embeddings for semantic memory retrieval.
# Pillow         -> Image-processing dependency required by some packages.
!pip install -q "crewai[litellm]" "langgraph" "pydantic-ai" "mem0ai" "groq" "sentence-transformers" "chromadb" "Pillow"

## Import Libraries and Configure the API

In [2]:
import os
from pathlib import Path
from typing import Literal, TypedDict, Any

from google.colab import userdata

# --------------------------------------------------
# Groq API key
# --------------------------------------------------

GROQ_API_KEY = userdata.get("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise RuntimeError(
        "Create a Colab Secret key named GROQ_API_KEY before running this notebook."
    )

os.environ["GROQ_API_KEY"] = GROQ_API_KEY


# --------------------------------------------------
# Models
# --------------------------------------------------
# Model used by CrewAI agents.
# CrewAI accesses Groq through LiteLLM.
CREWAI_MODEL = "groq/openai/gpt-oss-120b"

MEM0_MODEL = "openai/gpt-oss-20b"

# Model used by PydanticAI for structured-output validation.
PYDANTICAI_MODEL = "openai/gpt-oss-20b"

# Hugging Face embedding model used by Mem0.
# It converts text into numerical vectors for semantic search.
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

# Groq's OpenAI-compatible API endpoint.
# This allows PydanticAI and Mem0 to communicate with Groq using the OpenAI-compatible interface.
GROQ_BASE_URL = "https://api.groq.com/openai/v1"


print("Groq credentials loaded.")
print("CrewAI model:", CREWAI_MODEL)
print("Mem0 model:", MEM0_MODEL)
print("PydanticAI model:", PYDANTICAI_MODEL)
print("Embedding model:", EMBEDDING_MODEL)

Groq credentials loaded.
CrewAI model: groq/openai/gpt-oss-120b
Mem0 model: openai/gpt-oss-20b
PydanticAI model: openai/gpt-oss-20b
Embedding model: sentence-transformers/all-MiniLM-L6-v2


In [3]:
# Import Python's warnings module to control non-critical warning messages displayed in Colab
import warnings

warnings.filterwarnings(
    "ignore",
    category=DeprecationWarning,
    module="crewai"
)

warnings.filterwarnings(
    "ignore",
    category=DeprecationWarning,
    module="jupyter_client"
)

In [4]:
import mem0
import importlib.metadata

print("Mem0 version:", importlib.metadata.version("mem0ai"))

Mem0 version: 2.0.5


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## CrewAI/Groq compatibility

CrewAI currently has an open issue where cache_breakpoint can be added to messages sent to non-Anthropic providers such as Groq.

In [5]:
# ============================================================
# CrewAI + Groq compatibility patch
# ============================================================

import crewai.llms.cache as _crewai_cache

# Provide a compatibility fallback for the current CrewAI/LiteLLM combination when cache_breakpoint is not supported.
# This prevents the "cache_breakpoint is unsupported" error without changing the actual agent behavior.
_crewai_cache.mark_cache_breakpoint = lambda msg: msg

print("CrewAI/Groq cache_breakpoint compatibility patch applied.")

CrewAI/Groq cache_breakpoint compatibility patch applied.


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Define Pydantic Schemas

In [6]:
from pydantic import BaseModel, ConfigDict, Field

# Reject fields that are not explicitly defined
# This prevents agents from returning unexpected fields
class StrictModel(BaseModel):
    model_config = ConfigDict(
        extra="forbid",
        # Enforce strict type validation
        strict=True,
    )


class PlannerOutput(StrictModel):
  #  The overall goal generated by the Planner
    goal: str = Field(min_length=1)
    # Ordered execution steps.
    # At least one and at most eight steps are allowed.
    steps: list[str] = Field(
        min_length=1,
        max_length=8,
    )


class ExecutorOutput(StrictModel):
   # The actual deliverable produced by the Executor
    result: str = Field(
        # Minimum length prevents empty/placeholder responses
        # Maximum length keeps responses bounded
        min_length=20,
        max_length=4000,
        description=(
            "REQUIRED. The complete actual deliverable requested "
            "by the user. Do not summarize the deliverable. "
            "For a checklist, provide the complete checklist. "
            "For code, provide the requested code. "
            "For instructions, provide the actual instructions."
        ),
    )
    # Record which planner steps were completed
    completed_steps: list[str] = Field(
        min_length=1,
        max_length=8,
        description="REQUIRED. The planner steps that were actually completed.",
    )

class ReviewerOutput(StrictModel):
  # Reviewer can only approve or request a retry
    decision: Literal["approve", "retry"]
     # Short explanation of the review decision
    feedback: str = Field(min_length=1)


print("Pydantic schemas created.")

Pydantic schemas created.


## Configuring PydanticAI Model and Validators

For Groq's OpenAI-compatible endpoint, the current PydanticAI documentation supports OpenAIChatModel with OpenAIProvider(base_url=..., api_key=...).

In [7]:
# ============================================================
# PydanticAI configuration
# ============================================================

from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

# Create an OpenAI-compatible provider configured to use Groq
pydantic_provider = OpenAIProvider(
    # Groq exposes an OpenAI-compatible API endpoint
    base_url=GROQ_BASE_URL,
    # Authenticate using our Groq API key
    api_key=GROQ_API_KEY,
)
# Create the model that PydanticAI will use
pydantic_model = OpenAIChatModel(
    PYDANTICAI_MODEL,
    provider=pydantic_provider,
)

print("PydanticAI model configured.")


# Validator for Planner responses
planner_validator = Agent(
    pydantic_model,
    # Require output that follows PlannerOutput
    output_type=PlannerOutput,
    # Instructions specifically designed to produce concise structured output
    instructions=(
        "Return exactly two fields: goal and steps. "
        "The goal must be concise. "
        "Create 4 to 8 short actionable steps. "
        "Each step should be one concise sentence. "
        "Do not include explanations or long descriptions."
    ),
)

# Validator for Executor responses
executor_validator = Agent(
    pydantic_model,
    output_type=ExecutorOutput,
    instructions=(
        "You MUST return both fields: result and completed_steps. "

        "The result field is REQUIRED and MUST contain the "
        "complete actual deliverable requested by the user. "

        "The completed_steps field is REQUIRED and MUST contain "
        "the steps completed by the Executor. "

        "NEVER omit the result field. "

        "If the user asks for a checklist, result MUST contain "
        "the complete checklist itself, not a summary of the checklist. "

        "If the user asks for code, result MUST contain the code. "

        "If the user asks for steps, result MUST contain the actual steps. "

        "Keep the result concise but complete. "

        "Return only the fields required by ExecutorOutput."
    ),
)

# Validator for Reviewer responses
reviewer_validator = Agent(
    pydantic_model,
     # Require output that follows ReviewerOutput
    output_type=ReviewerOutput,
     # Restrict the reviewer to approve/retry decisions
    instructions=(
        "Validate and structure the reviewer response. "
        "The decision must be exactly 'approve' or 'retry'. "
        "Return only the fields defined by ReviewerOutput."
    ),
)


print("PydanticAI validators created.")

PydanticAI model configured.
PydanticAI validators created.


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


PydanticAI supports Pydantic models directly as output_type, with the resulting .output containing the validated model.

## Mem0 Persistent Memory Configuration



In [8]:
# ============================================================
# Mem0 configuration
# ============================================================

# Import Path so we can create persistent directories for the memory database and vector store
from mem0 import Memory


MEMORY_DIR = Path("/content/guardrailed_mem0")
MEMORY_DIR.mkdir(parents=True, exist_ok=True)
# SQLite database stores Mem0's memory/history metadata
MEMORY_DB = MEMORY_DIR / "history.sqlite3"
# Chroma stores vector representations used for semantic search
CHROMA_DIR = MEMORY_DIR / "chroma"

# Configure Mem0's language model, embedding model, vector database, and persistent history database
mem0_config = {
    "llm": {
        "provider": "openai",
        "config": {
            "model": "openai/gpt-oss-20b",
            "api_key": GROQ_API_KEY,
            "openai_base_url": GROQ_BASE_URL,
            "temperature": 0.0,  # Deterministic memory behavior
            "max_tokens": 512,   # Keep Mem0's internal response size bounded
        },
    },
    # Configure the embedding model used for semantic search
    "embedder": {
        "provider": "huggingface",
        "config": {
            "model": EMBEDDING_MODEL,
        },
    },
    # Configure Chroma as Mem0's vector database
    "vector_store": {
        "provider": "chroma",
        "config": {
            # Store Chroma data in our persistent memory directory
            "path": str(CHROMA_DIR),
        },
    },
   # Store memory history in a local SQLite database
    "history_db_path": str(MEMORY_DB),
}

# Create the Mem0 memory instance using the configuration above
memory = Memory.from_config(mem0_config)

# Define the user/session identifier used to associate memories
# Changing this value creates a separate memory namespace
USER_ID = "colab-final-workflow-harshitha-mas-final"

print("Mem0 configured.")
print("Memory directory:", MEMORY_DIR)

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/mem0/embeddings/huggingface.py:27: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.config.embedding_dims = self.config.embedding_dims or self.model.get_sentence_embedding_dimension()


Mem0 configured.
Memory directory: /content/guardrailed_mem0


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [9]:
TEST_USER_ID = "mem0-persistence-test-v3"


add_result = memory.add(
    [
        {
            "role": "user",
            "content": "I prefer concise release checklists."
        }
    ],
    user_id=TEST_USER_ID,
    infer=False,
)


print("Add result:")
print(add_result)

Add result:
{'results': [{'id': '85d6fcab-83e3-4599-885c-2c838bd65876', 'memory': 'I prefer concise release checklists.', 'event': 'ADD', 'actor_id': None, 'role': 'user'}]}


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [10]:
all_memories = memory.get_all(
    filters={
        "user_id": TEST_USER_ID
    }
)

print("All stored memories:")
print(all_memories)

All stored memories:
{'results': [{'id': '31ed68e9-41d7-4fe5-8bd9-1698c9d5b203', 'memory': 'I prefer concise release checklists.', 'hash': '664b5b0fd22b0dfe04643b24585ce441', 'metadata': None, 'created_at': '2026-09-29T02:58:51.127959+00:00', 'updated_at': '2026-09-29T02:58:51.127959+00:00', 'user_id': 'mem0-persistence-test-v3', 'role': 'user'}, {'id': '13288578-5349-485b-ba7d-42d28fd7c2fb', 'memory': 'I prefer concise release checklists.', 'hash': '664b5b0fd22b0dfe04643b24585ce441', 'metadata': None, 'created_at': '2026-09-29T03:03:08.287590+00:00', 'updated_at': '2026-09-29T03:03:08.287590+00:00', 'user_id': 'mem0-persistence-test-v3', 'role': 'user'}, {'id': '85d6fcab-83e3-4599-885c-2c838bd65876', 'memory': 'I prefer concise release checklists.', 'hash': '664b5b0fd22b0dfe04643b24585ce441', 'metadata': None, 'created_at': '2026-09-29T03:07:21.357214+00:00', 'updated_at': '2026-09-29T03:07:21.357214+00:00', 'user_id': 'mem0-persistence-test-v3', 'role': 'user'}]}


In [11]:
retrieved = memory.search(
    "concise release checklists",
    filters={
        "user_id": TEST_USER_ID
    }
)

print("Retrieved memories:")
print(retrieved)

Retrieved memories:
{'results': [{'id': '31ed68e9-41d7-4fe5-8bd9-1698c9d5b203', 'memory': 'I prefer concise release checklists.', 'hash': '664b5b0fd22b0dfe04643b24585ce441', 'metadata': None, 'score': 0.831236491094558, 'created_at': '2026-09-29T02:58:51.127959+00:00', 'updated_at': '2026-09-29T02:58:51.127959+00:00', 'user_id': 'mem0-persistence-test-v3', 'role': 'user'}, {'id': '13288578-5349-485b-ba7d-42d28fd7c2fb', 'memory': 'I prefer concise release checklists.', 'hash': '664b5b0fd22b0dfe04643b24585ce441', 'metadata': None, 'score': 0.831236491094558, 'created_at': '2026-09-29T03:03:08.287590+00:00', 'updated_at': '2026-09-29T03:03:08.287590+00:00', 'user_id': 'mem0-persistence-test-v3', 'role': 'user'}, {'id': '85d6fcab-83e3-4599-885c-2c838bd65876', 'memory': 'I prefer concise release checklists.', 'hash': '664b5b0fd22b0dfe04643b24585ce441', 'metadata': None, 'score': 0.831236491094558, 'created_at': '2026-09-29T03:07:21.357214+00:00', 'updated_at': '2026-09-29T03:07:21.357214+00

## MEM0 Memory helper functions

In [12]:
def get_memory_context(task: str, user_id: str) -> list[str]:
    """
    Retrieve relevant persistent memories for the current task.
    """
    # Search Mem0 for memories related to the current task
    results = memory.search(
        task,
        filters={
            "user_id": user_id
        },
    )

    # Mem0 may return a dictionary containing a "results" list
    if isinstance(results, dict):
        results = results.get("results", [])

    # Make sure the result is always handled as a list
    if not isinstance(results, list):
        results = [results]


    # Store the extracted memory text
    context = []

    # Process each retrieved memory
    for item in results:
      # Mem0 normally returns dictionaries
        if isinstance(item, dict):
          # Extract the actual memory text
            value = item.get("memory")
            # Only keep memories that contain text
            if value:
                context.append(str(value))
        # Also handle the case where Mem0 returns a plain string.
        elif isinstance(item, str):
            context.append(item)
    # Limit the number of memories passed to the agents - prevents unnecessary context/token growth
    return context[:5]


def save_memory(
    task: str,
    result: str,
    user_id: str,
):
    """
    Save the successful result of a workflow execution into Mem0.

    task:
        The original user request.

    result:
        The final answer produced by the Executor.

    user_id:
        Identifier used to associate the memory with a user/session.
    """

    # Combine the task and successful result into one memory record.
    memory_text = (
        f"Completed task: {task}\n"
        f"Successful result: {result}"
    )

    # Store the memory in Mem0

    # infer=False:
    # it prevents Mem0 from making an additional LLM call to extract/summarize the memory.
    # This reduces token usage and avoids large-request errors.
    return memory.add(
        [
            {
                "role": "user",
                "content": memory_text,
            }
        ],
        user_id=user_id,
        infer=False,
    )

## Creating the three CrewAI Agents

In [13]:
from crewai import Agent, Crew, Process, Task


def create_crewai_agent(
    role: str,
    goal: str,
    backstory: str,
):
    # Create and return a crew agent
    return Agent(
        # Role describes what the agent is responsible for
        role=role,
        # Goal defines the agent's main objective
        goal=goal,
        # Backstory provides behavioral context
        backstory=backstory,
        # Use the configured Groq model
        llm=CREWAI_MODEL,
        # Agents do not delegate work to additional agents.
        # This keeps the architecture limited to our three agents.
        allow_delegation=False,
        # Disable verbose CrewAI logging
        verbose=False,
    )

# Create the Planner agent.
#
# Its responsibility is to understand the user's task
# and convert it into a short actionable plan.
planner_agent = create_crewai_agent(
    role="Planner",
    goal="Break the user's task into a short, precise ordered plan.",
    backstory=(
        "You analyze the user's task and create an actionable plan "
        "without inventing requirements."
    ),
)

# Create the Executor agent.
#
# Its responsibility is to perform the task
# and produce the actual requested deliverable.
executor_agent = create_crewai_agent(
    role="Executor",
    goal=(
        "Complete the user's task and produce the actual requested "
        "deliverable concisely."
    ),
    backstory=(
        "You are the execution agent in a multi-agent workflow. "
        "The Planner has already analyzed the task. "
        "Produce the actual requested deliverable. "
        "Do not merely describe the plan. "
        "Keep responses concise and directly usable."
    ),
)

# Create the Reviewer agent.
#
# Its responsibility is to check whether the Executor
# actually completed the requested task.
reviewer_agent = create_crewai_agent(
    role="Reviewer",
    goal=(
        "Check whether the Executor actually completed the user's "
        "request and whether the resulting deliverable is usable."
    ),
    backstory=(
        "You are a quality-control agent in a multi-agent workflow. "
        "Approve responses that actually answer the user's request "
        "and contain the requested deliverable. "
        "Request a retry only when the deliverable is missing, "
        "incomplete, or merely describes what should be done."
    ),
)

print("CrewAI agents created.")

CrewAI agents created.


/usr/local/lib/python3.13/dist-packages/crewai/agent/core.py:431: DeprecationWarning: function_calling_llm is deprecated and will be removed in a future release.
  if self.function_calling_llm and not isinstance(
/usr/local/lib/python3.13/dist-packages/crewai/agent/core.py:439: DeprecationWarning: deprecated
  if self.allow_code_execution:
/usr/local/lib/python3.13/dist-packages/crewai/agent/core.py:450: DeprecationWarning: deprecated
  if self.reasoning and self.planning_config is None:
/usr/local/lib/python3.13/dist-packages/crewai/agent/core.py:431: DeprecationWarning: function_calling_llm is deprecated and will be removed in a future release.
  if self.function_calling_llm and not isinstance(
/usr/local/lib/python3.13/dist-packages/crewai/agent/core.py:439: DeprecationWarning: deprecated
  if self.allow_code_execution:
/usr/local/lib/python3.13/dist-packages/crewai/agent/core.py:450: DeprecationWarning: deprecated
  if self.reasoning and self.planning_config is None:
/usr/local/lib

## Run CrewAI Agents Asynchronously

In [14]:
async def run_crewai_agent(
    agent,
    description: str,
) -> str:
    """
    Run one CrewAI agent asynchronously and return its raw output.
    """
     # Create a task for the selected agent
    task = Task(
        # Instructions given to the agent
        description=description,
        # Tell CrewAI what kind of output is expected
        expected_output=(
            "Return the requested result clearly and concisely."
        ),
        # Assign the task to the provided agent
        agent=agent,
    )

# Create a one-agent Crew
    crew = Crew(
        agents=[agent],
        tasks=[task],
         # Execute the task sequentially
        process=Process.sequential,
        # Keep CrewAI logs concise
        verbose=False,
    )

    result = await crew.kickoff_async()
    # Return the raw text produced by the agent
    return result.raw

## PydanticAI validation helper

In [15]:
async def validate_with_pydantic_ai(
    validator,
    raw_output: str,
):
    """
    Send raw agent output through PydanticAI and return validated structured data.
    """
    # Ask PydanticAI to parse and validate the raw output
    result = await validator.run(raw_output)
    # Extract the structured Pydantic object
    output = getattr(result, "output", None)
    # Fail explicitly if structured output was not returned
    if output is None:
        raise ValueError(
            "PydanticAI returned no structured output."
        )
    # Return the validated Pydantic object
    return output

## Execution Guardrail Configuration

In [17]:
# Maximum number of times the workflow can execute
MAX_EXECUTIONS = 3

# Safe response returned when the retry budget is exhausted
SAFE_FALLBACK = (
    "Execution stopped safely after the maximum retry "
    "budget was reached."
)

## Execution Counter

In [21]:
def increment_execution(state: GraphState):
    """
    Increment the workflow execution counter
    """

    # Read the current counter
    # If it does not exist, start from zero
    current_count = state.get(
        "execution_count",
        0,
    )

    # Increment the counter by one
    return {
        "execution_count": current_count + 1
    }

## Routing based on Execution Budget

In [22]:
def route_after_increment(state: GraphState):
    """
    Decide whether the workflow can continue or must activate the kill-switch
    """
    # If the execution count exceeds the allowed budget, terminate safely through the fallback node.
    if state["execution_count"] >= MAX_EXECUTIONS:
        return "fallback"
 # Otherwise continue with the Planner
    return "planner"


def route_after_review(state: GraphState):
    """
    Decide whether the workflow is complete or needs another attempt
    """

    # If the Reviewer approves the result, the workflow terminates successfully.

    if state["review"].decision == "approve":
        return "complete"
    # If the Reviewer requests a retry, return to the execution loop.
    return "retry"

### Safe Fallback

In [23]:
def fallback_node(state: GraphState):
    """
    Kill-switch node used when the retry budget is exceeded.
    """
    return {
        "final_response": SAFE_FALLBACK
    }

## Define LangGraph State

In [16]:
from typing import TypedDict

# ============================================================
# LANGGRAPH STATE
# ============================================================
#
# GraphState defines all information that can move between
# Planner, Executor, Reviewer, and the safety/guardrail nodes.
#
# LangGraph uses this shared state to keep track of:
# - the original user task
# - persistent memory
# - execution/retry count
# - structured agent outputs
# - final response
# ============================================================
class GraphState(TypedDict, total=False):

    # Identifier used to retrieve and store persistent memory.
    user_id: str

    # Original task submitted by the user.
    task: str

    # Number of execution attempts.
    execution_count: int

    # Relevant memories retrieved from Mem0.
    memory_context: list[str]

    # Structured Planner output.
    plan: PlannerOutput

    # Structured Executor output.
    execution: ExecutorOutput

    # Structured Reviewer output.
    review: ReviewerOutput

    # Final response returned by the workflow.
    final_response: str

## Planner Node

In [18]:
async def planner_node(state):
  # Run the planner and validate its output


    # Retrieve relevant memories from the graph state
    memory_context = "\n".join(state.get("memory_context", []))
    # If no memory was found, explicitly tell the Planner
    if not memory_context:
        memory_context = "No relevant persistent memory was found."
    # Run the CrewAI Planner
    raw_output = await run_crewai_agent(
        planner_agent,
        f"""
User task:
{state["task"]}

Relevant persistent memory:
{memory_context}

Create a concise execution plan.

Requirements:
- Provide one clear goal.
- Provide 4 to 8 short actionable steps.
- Keep each step concise.
- Do not provide the final answer.
- Do not add unnecessary explanations.
""",
    )
    # Validate the raw Planner response against the strict PydanticAI schema
    plan = await validate_with_pydantic_ai(
        planner_validator,
        raw_output
    )
    # Store the validated plan in LangGraph state
    return {"plan": plan}

## Executor Node

In [19]:
async def executor_node(state: GraphState):
  # Run the Executor using the Planner's structured output

  # Run the CrewAI Executor
    raw_output = await run_crewai_agent(
        executor_agent,
        f"""
Complete the user's task directly.

USER TASK:
{state["task"]}

PLANNER GOAL:
{state["plan"].goal}

PLANNER STEPS:
{chr(10).join(
    f"{i + 1}. {step}"
    for i, step in enumerate(state["plan"].steps)
)}

IMPORTANT OUTPUT REQUIREMENTS:

1. You MUST provide the `result` field.
2. You MUST provide the `completed_steps` field.
3. The `result` field MUST contain the actual final deliverable.
4. Do NOT use `result` as a summary of what you did.
5. Do NOT omit `result`.
6. The `completed_steps` field must list the planner steps
   that were actually completed.

For example:

If the user asks for a checklist:
result = the complete checklist itself.

If the user asks for code:
result = the complete requested code.

If the user asks for instructions:
result = the actual instructions.

Do not merely describe the plan.
Do not say the task is ready for execution.
Do not return only completed_steps.

Return the actual deliverable in `result`.
Keep the answer concise but complete.
""",
    )
    # Validate the Executor's raw response
    execution = await validate_with_pydantic_ai(
        executor_validator,
        raw_output
    )
    # Store the validated execution result in graph state
    return {"execution": execution}

## Reviewer Node

In [20]:
async def reviewer_node(state: GraphState):

    # Review the Executor's output and decide whether to approve it or request another attempt
       # Run the CrewAI Reviewer
    raw_output = await run_crewai_agent(
        reviewer_agent,
        f"""
You are the final quality-control reviewer.

ORIGINAL USER TASK:
{state["task"]}

PLANNER:
{state["plan"].model_dump_json()}

EXECUTOR:
{state["execution"].model_dump_json()}

Determine whether the Executor actually completed the user's request.

APPROVE when:
- the actual requested deliverable is present
- the response directly answers the user's task
- the major requirements from the planner are covered
- the result is usable by the user

RETRY only when:
- the actual deliverable is missing
- the response is only a plan or description
- the response does not answer the user's task
- major planner requirements are missing

Do NOT reject the response merely because:
- completed_steps uses slightly different wording
- the Executor used a different but reasonable format
- the response contains more detail than the planner

The user's task is:
"{state["task"]}"

Return either approve or retry with concise feedback.
""",
    )
    # Validate the Reviewer response with PydanticAI
    review = await validate_with_pydantic_ai(
        reviewer_validator,
        raw_output
    )
    # If the Reviewer approves the result, save the successful result to persistent memory.
    if review.decision == "approve":
        save_memory(
            task=state["task"],
            result=state["execution"].result,
            user_id=state["user_id"],
        )
        # Store both the review and final response
        return {
            "review": review,
            "final_response": state["execution"].result,
        }
    # If retry is requested, only update the review
    # LangGraph will route back to the execution counter
    return {"review": review}

## Build the LangGraph

In [24]:
from langgraph.graph import StateGraph, START, END
# ============================================================
# BUILD LANGGRAPH
# ============================================================
#
# This creates the actual executable graph:
#
# START
#   ↓
# Execution Counter
#   ↓
# Planner
#   ↓
# Executor
#   ↓
# Reviewer
#   ├── APPROVE → END
#   └── RETRY → Execution Counter
#
# If execution_count > 3:
# Execution Counter → Fallback → END
# ============================================================

# Build and compile the Planner → Executor → Reviewer graph
def build_graph():
    # Create a LangGraph StateGraph using our GraphState definition
    graph = StateGraph(GraphState)

    # --------------------------------------------------------
    # Register workflow nodes
    # --------------------------------------------------------
    # Execution counter/guardrail
    graph.add_node(
        "increment_execution",
        increment_execution,
    )

    # Planner agent
    graph.add_node(
        "planner",
        planner_node,
    )
    # Executor agent
    graph.add_node(
        "executor",
        executor_node,
    )
    # Reviewer agent
    graph.add_node(
        "reviewer",
        reviewer_node,
    )
    # Safe fallback / Kill-switch
    graph.add_node(
        "fallback",
        fallback_node,
    )

    # Start every workflow by incrementing the execution counter
    # --------------------------------------------------------
    # START → Execution Counter
    # --------------------------------------------------------
    graph.add_edge(
        START,
        "increment_execution",
    )

    # After incrementing, check the execution budget
    # Counter → planner OR fallback
    # --------------------------------------------------------
    # Execution Counter → Planner OR Fallback
    # --------------------------------------------------------
    #
    # This is the first safety decision.
    #
    # execution_count <= 3 → Planner
    # execution_count > 3  → Fallback
    # --------------------------------------------------------

    graph.add_conditional_edges(
        "increment_execution",
        route_after_increment,
        {
            # Normal execution path
            "planner": "planner",
            # Kill-switch path
            "fallback": "fallback",
        },
    )

    # Normal agent pipeline:
    # Planner → Executor → Reviewer
    # --------------------------------------------------------
    # Main agent pipeline
    # --------------------------------------------------------
    # Planner → Executor
    graph.add_edge(
        "planner",
        "executor",
    )

    # Executor → Reviewer
    graph.add_edge(
        "executor",
        "reviewer",
    )

 # Reviewer decides whether to finish or retry
    # Reviewer → END OR retry
    # --------------------------------------------------------
    # Reviewer → END OR RETRY
    # --------------------------------------------------------
    #
    # APPROVE → workflow completes
    # RETRY   → execution counter checks the budget again
    # --------------------------------------------------------
    graph.add_conditional_edges(
        "reviewer",
        route_after_review,
        {
            # Approved result ends the workflow
            "complete": END,
            # Rejected result starts another execution attempt
            "retry": "increment_execution",
        },
    )

 # The fallback node always terminates the workflow
    # --------------------------------------------------------
    # Fallback → END
    # --------------------------------------------------------
    graph.add_edge(
        "fallback",
        END,
    )
# Compile the graph into an executable application
    return graph.compile()

# Build and compile the LangGraph workflow
app = build_graph()

print("LangGraph workflow compiled successfully.")

LangGraph workflow compiled successfully.


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Main Workflow Function


In [25]:
async def run_workflow(
    task: str,
    user_id: str = USER_ID,
):
    # Execute the complete multi-agent workgflow

    # Retrieve persistent memory before planning
    # Search persistent Mem0 memory before starting the workflow
    # This allows previous successful results to influence the current Planner without passing the entire history.
    memory_context = get_memory_context(
        task,
        user_id,
    )
    # Create the initial LangGraph state
    initial_state: GraphState = {
        "task": task,
        "user_id": user_id,
        # Start with zero executions
        "execution_count": 0,
        # Provide relevant memories to the graph
        "memory_context": memory_context,
    }
    # Execute the graph asynchronously
    result = await app.ainvoke(
        initial_state,
        # LangGraph's recursion limit is a secondary safety mechanism.
        # The application's own execution counter remains the primary retry guardrail.
        config={
            "recursion_limit": 25
        },
    )
    # Return the complete final graph state
    return result

## Executing the workflow

In [26]:
# Run the complete Planner → Executor → Reviewer workflow with user's task
result = await run_workflow(
    "Create a concise checklist for deploying a Python API."
)

print("Final response:")
print(result["final_response"])

/usr/local/lib/python3.13/dist-packages/crewai/crew.py:650: DeprecationWarning: function_calling_llm is deprecated and will be removed in a future release.
  if self.function_calling_llm and not isinstance(self.function_calling_llm, LLM):
/usr/local/lib/python3.13/dist-packages/crewai/telemetry/telemetry.py:433: DeprecationWarning: function_calling_llm is deprecated and will be removed in a future release.
  if getattr(agent, "function_calling_llm", None)
/usr/local/lib/python3.13/dist-packages/crewai/telemetry/telemetry.py:440: DeprecationWarning: deprecated
  "allow_code_execution?": getattr(
/usr/local/lib/python3.13/dist-packages/crewai/crews/utils.py:372: DeprecationWarning: function_calling_llm is deprecated and will be removed in a future release.
  crew.function_calling_llm,
/usr/local/lib/python3.13/dist-packages/crewai/crews/utils.py:87: DeprecationWarning: function_calling_llm is deprecated and will be removed in a future release.
  if not agent.function_calling_llm:  # type

Final response:
### Checklist for Deploying a Python API

1. **Select Deployment Target**
   - Choose between Docker, a cloud platform (e.g., AWS Elastic Beanstalk, GCP Cloud Run, Azure App Service), or a traditional VM.

2. **Prepare Runtime Environment**
   - Install required Python version.
   - Create and activate a virtual environment (`python -m venv venv`).
   - Install dependencies (`pip install -r requirements.txt`).
   - Install and configure a WSGI/ASGI server (e.g., Gunicorn, Uvicorn, Daphne).

3. **Configure Settings & Secrets**
   - Define environment variables for configuration (e.g., `APP_ENV`, `DEBUG`).
   - Store secrets securely (AWS Secrets Manager, GCP Secret Manager, Vault, or `.env` with proper permissions).
   - Set database connection URLs and any external service endpoints.

4. **Build & Deploy**
   - **Docker**: Write a `Dockerfile`, build the image (`docker build -t myapi:latest .`), test locally, push to registry (`docker push <registry>/myapi:latest`).
   

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Inspecting the complete workflow state

In [27]:
result = await run_workflow(
    "Create a concise checklist for deploying a Python API."
)
# Display how many execution attempts were required.
print("Execution count:")
print(result.get("execution_count"))

# Display relevant memories retrieved before execution.
print("\nMemory context:")
print(result.get("memory_context"))

# Display the structured Planner output
print("\nPlan:")
print(result.get("plan"))

# Display the structured Executor output
print("\nExecution:")
print(result.get("execution"))

# Display the Reviewer decision
print("\nReview:")
print(result.get("review"))

# Display the final approved response
print("\nFinal response:")
print(result.get("final_response"))

/usr/local/lib/python3.13/dist-packages/crewai/crew.py:650: DeprecationWarning: function_calling_llm is deprecated and will be removed in a future release.
  if self.function_calling_llm and not isinstance(self.function_calling_llm, LLM):
/usr/local/lib/python3.13/dist-packages/crewai/telemetry/telemetry.py:433: DeprecationWarning: function_calling_llm is deprecated and will be removed in a future release.
  if getattr(agent, "function_calling_llm", None)
/usr/local/lib/python3.13/dist-packages/crewai/telemetry/telemetry.py:440: DeprecationWarning: deprecated
  "allow_code_execution?": getattr(
/usr/local/lib/python3.13/dist-packages/crewai/crews/utils.py:372: DeprecationWarning: function_calling_llm is deprecated and will be removed in a future release.
  crew.function_calling_llm,
/usr/local/lib/python3.13/dist-packages/crewai/crews/utils.py:87: DeprecationWarning: function_calling_llm is deprecated and will be removed in a future release.
  if not agent.function_calling_llm:  # type

Execution count:
1

Memory context:
['Completed task: Create a concise checklist for deploying a Python API.\nSuccessful result: ### Checklist for Deploying a Python API\n\n1. **Select Deployment Target**\n   - Choose between Docker, a cloud platform (e.g., AWS Elastic Beanstalk, GCP Cloud Run, Azure App Service), or a traditional VM.\n\n2. **Prepare Runtime Environment**\n   - Install required Python version.\n   - Create and activate a virtual environment (`python -m venv venv`).\n   - Install dependencies (`pip install -r requirements.txt`).\n   - Install and configure a WSGI/ASGI server (e.g., Gunicorn, Uvicorn, Daphne).\n\n3. **Configure Settings & Secrets**\n   - Define environment variables for configuration (e.g., `APP_ENV`, `DEBUG`).\n   - Store secrets securely (AWS Secrets Manager, GCP Secret Manager, Vault, or `.env` with proper permissions).\n   - Set database connection URLs and any external service endpoints.\n\n4. **Build & Deploy**\n   - **Docker**: Write a `Dockerfile

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Demonstrating Persistent Memory

In [28]:
# Retrieve all memories associated with the current user ID.
# This demonstrates that successful workflow results are being stored.

stored = memory.get_all(
    filters={"user_id": USER_ID}
)
# Display the stored memories.
print("\nStored Mem0 memories:")
print(stored)


Stored Mem0 memories:
{'results': [{'id': 'e9ad6af2-bc4f-4dac-a76d-fda6931f7cf9', 'memory': 'Completed task: Create a concise checklist for deploying a Python API.\nSuccessful result: ### Checklist for Deploying a Python API\n\n1. **Select Deployment Target**\n   - Choose between Docker, a cloud platform (e.g., AWS Elastic Beanstalk, GCP Cloud Run, Azure App Service), or a traditional VM.\n\n2. **Prepare Runtime Environment**\n   - Install required Python version.\n   - Create and activate a virtual environment (`python -m venv venv`).\n   - Install dependencies (`pip install -r requirements.txt`).\n   - Install and configure a WSGI/ASGI server (e.g., Gunicorn, Uvicorn, Daphne).\n\n3. **Configure Settings & Secrets**\n   - Define environment variables for configuration (e.g., `APP_ENV`, `DEBUG`).\n   - Store secrets securely (AWS Secrets Manager, GCP Secret Manager, Vault, or `.env` with proper permissions).\n   - Set database connection URLs and any external service endpoints.\n\n4. 

## Demonstrate Memory Retrieval in a New Execution

In [29]:
# Use a similar task to test whether Mem0 can retrieve
# information from a previous workflow execution.
follow_up_task = (
    "Give me the Python API deployment checklist again."
)

# Search persistent semantic memory for relevant information.
retrieved_memory = get_memory_context(
    follow_up_task,
    USER_ID,
)

# Display the retrieved memories.
print("Retrieved memory:")
for item in retrieved_memory:
    print("-", item)

Retrieved memory:
- Completed task: Create a concise checklist for deploying a Python API.
Successful result: ### Checklist for Deploying a Python API

1. **Select Deployment Target**
   - Choose between Docker, a cloud platform (e.g., AWS Elastic Beanstalk, GCP Cloud Run, Azure App Service), or a traditional VM.

2. **Prepare Runtime Environment**
   - Install required Python version.
   - Create and activate a virtual environment (`python -m venv venv`).
   - Install dependencies (`pip install -r requirements.txt`).
   - Install and configure a WSGI/ASGI server (e.g., Gunicorn, Uvicorn, Daphne).

3. **Configure Settings & Secrets**
   - Define environment variables for configuration (e.g., `APP_ENV`, `DEBUG`).
   - Store secrets securely (AWS Secrets Manager, GCP Secret Manager, Vault, or `.env` with proper permissions).
   - Set database connection URLs and any external service endpoints.

4. **Build & Deploy**
   - **Docker**: Write a `Dockerfile`, build the image (`docker build -t

Demonstrating an explicit kill-switch test

In [30]:
# ============================================================
# TEST KILL-SWITCH
# ============================================================
#
# This test directly verifies that the workflow routes to
# the fallback when execution_count exceeds MAX_EXECUTIONS.
# ============================================================

# Simulate a state where the maximum execution budget
# has already been exceeded.
test_state = {
    "execution_count": 4
}

# Ask the router where the workflow should go.
route = route_after_increment(test_state)

print("Execution count:", test_state["execution_count"])
print("Maximum allowed:", MAX_EXECUTIONS)
print("Route:", route)

Execution count: 4
Maximum allowed: 3
Route: fallback


In [31]:
# Execute the fallback node directly.
fallback_result = fallback_node(test_state)

print("\nFallback response:")
print(fallback_result["final_response"])


Fallback response:
Execution stopped safely after the maximum retry budget was reached.
